# Corrected bake-off + ADT (exposure)

One consolidated input now carries every feature. Tests four cumulative feature sets on the
`<5-crash` candidate set with genuine 5-fold out-of-fold scoring, plus a **rate model** (ADT as a
Tweedie offset — predicts KSI *per unit of traffic*, surfacing design-dangerous sites over merely
busy ones):

- **A** crash-only  ·  **D** + infrastructure  ·  **E** + spatial  ·  **F** + **ADT** (exposure)
- **F-rate** — E features, ADT as offset (crash rate, not count)

### Upload 2 files
| file | in |
|---|---|
| `verified_model_input.parquet` | `data\\model\\` |
| `frozen_params.json` | `data\\model\\` |


In [ ]:
!pip -q install xgboost 2>/dev/null
import xgboost; print("xgboost", xgboost.__version__, "| ready")


In [ ]:
from google.colab import files
up = files.upload()
print("MISSING:", ({"verified_model_input.parquet","frozen_params.json"} - set(up)) or "none")


In [ ]:
import json, numpy as np, pandas as pd, xgboost as xgb
from scipy import stats
from sklearn.model_selection import GroupKFold, StratifiedKFold

COST=5_175_524; EFF=0.30; PROG=3_200_000; SEED=42; NF=5; CITY_MIN=5
CRASH=["crashes_36mo","crashes_72mo","ped_crashes_72mo","bike_crashes_72mo","broadside_72mo",
 "left_turn_72mo","dui_72mo","night_72mo","ped_row_violation_72mo","years_since_last_crash",
 "distinct_crash_days_72mo","worst_severity_72mo","crash_trend_slope","emergence_velocity",
 "emergence_acceleration","mann_kendall_tau","changepoint_prob","ewma_crashes","momentum_ratio","covid_period_share"]
SPAT=["nbr_crashes_150m","nbr_crashes_400m","nbr_ksi_400m","node_density_400m","n_hotspots_400m","dist_nearest_hotspot_m"]
ADT=["log_aadt","aadt_missing"]
NON=set(CRASH+SPAT+ADT+["intersection_id","KSI_label","spatial_block","persistence_baseline_score","crashes_feat","aadt"])

df=pd.read_parquet("verified_model_input.parquet")
INFRA=[c for c in df.columns if c not in NON]
corr=df[df["crashes_feat"]<CITY_MIN].reset_index(drop=True)
print(f"corrected candidates: {len(corr)} | pos>=1={int((corr.KSI_label>=1).sum())} | ADT present {int((corr.aadt_missing==0).sum())}\n")
SETS={"A_crash":CRASH,"D_crash_infra":CRASH+INFRA,"E_crash_infra_spatial":CRASH+INFRA+SPAT,"F_plus_ADT":CRASH+INFRA+SPAT+ADT}

fp=json.load(open("frozen_params.json"))["frozen"]
params=dict(objective="reg:tweedie",tweedie_variance_power=float(fp["tweedie_variance_power"]),
    max_depth=int(fp["max_depth"]),learning_rate=float(fp["learning_rate"]),n_estimators=int(fp["n_estimators"]),
    reg_alpha=float(fp["reg_alpha"]),reg_lambda=float(fp["reg_lambda"]),min_child_weight=int(fp["min_child_weight"]),
    subsample=float(fp["subsample"]),colsample_bytree=float(fp["colsample_bytree"]),random_state=42,verbosity=0)

y=corr.KSI_label.values.astype(float); groups=corr.spatial_block.values; base=corr.persistence_baseline_score.values
offset=corr["log_aadt"].values.astype(float)
def folds(m):
    if m=="random": return list(StratifiedKFold(NF,shuffle=True,random_state=SEED).split(y,(y>=2).astype(int)))
    return list(GroupKFold(NF).split(y,y,groups=groups))
def oof(X,m,use_offset=False):
    o=np.full(len(y),np.nan)
    for tr,te in folds(m):
        mm=xgb.XGBRegressor(**params)
        if use_offset: mm.fit(X[tr],y[tr],base_margin=offset[tr]); o[te]=mm.predict(X[te],base_margin=np.zeros(len(te)))
        else: mm.fit(X[tr],y[tr]); o[te]=mm.predict(X[te])
    return o
def money(s,T=1,K=500):
    kl=y[np.argsort(-s)[:K]]; ev=int(kl[kl>=T].sum()); return ev,ev*COST*EFF
def sp(s): return float(stats.spearmanr(s,y).correlation)

print("========== corrected set — vs baseline (city=$0 here) ==========")
print(f"{'baseline':24s} rho={sp(base):+.4f} | >=1 ${money(base)[1]/1e6:.1f}M ({money(base)[0]}ev)\n")
res={}; store={}
for sn,cols in SETS.items():
    X=corr[cols].fillna(0.0).values.astype(float)
    for mode in ["random","spatial"]:
        o=oof(X,mode); store[(sn,mode)]=o; me,md=money(o); be,bd=money(base)
        print(f"{sn:24s} {mode:7s} rho={sp(o):+.4f} | >=1 ${md/1e6:5.1f}M({me}ev)  vs-base {(md-bd)/1e6:+.1f}M")
        res[f"{sn}__{mode}"]={"spearman":round(sp(o),4),"ge1_$M":round(md/1e6,1),"ge1_vs_base_$M":round((md-bd)/1e6,1)}
    print()
# rate model: E features, ADT as Tweedie offset -> ranks by KSI per unit traffic
XE=corr[SETS["E_crash_infra_spatial"]].fillna(0.0).values.astype(float)
for mode in ["random","spatial"]:
    o=oof(XE,mode,use_offset=True); me,md=money(o); be,bd=money(base)
    print(f"{'F_rate (E, ADT offset)':24s} {mode:7s} rho={sp(o):+.4f} | >=1 ${md/1e6:5.1f}M({me}ev)  vs-base {(md-bd)/1e6:+.1f}M")
    res[f"F_rate__{mode}"]={"spearman":round(sp(o),4),"ge1_$M":round(md/1e6,1),"ge1_vs_base_$M":round((md-bd)/1e6,1)}

json.dump(res,open("adt_bakeoff_results.json","w"),indent=2)
print("\nSaved adt_bakeoff_results.json — send me this.")


## What to watch
- **F vs E**: does adding ADT as a feature lift the dollars/rho beyond E? And does infra's SHAP
  weight drop (ADT absorbing the exposure that road-class was faking)?
- **F_rate**: ranks by KSI *per unit of traffic*. If it holds recall while re-ordering toward
  quieter-but-worse-designed sites, that's the differentiator the City's count/feature programs
  don't produce. If its recall collapses, exposure was most of the signal.

Paste me `adt_bakeoff_results.json` and I'll read it.
